# 03 - Data Preprocessing (PhishGuard)

**Input:** *cleaned.csv*

**Output:** *cleaned_with_domains.csv.gz*

This notebook performs preprocessing and registered-domain analysis on the cleaned PhiUSIIL Phishing URL Dataset.

The main objective is to extract registered domains from URLs, examine domain repetition and class overlap, and prepare the dataset for the domain-disjoint evaluation planned in Phase 2.

Only URL-based features are retained for the primary detection models. No webpage content is fetched or analyzed.

In [52]:
# Import libraries

import os
import ipaddress                            # for handling IP address URLs

import pandas as pd                         # for data manipulation
import tldextract                           # for extracting registered domains

from urllib.parse import urlsplit           # for extracting hostnames from URLs

pd.set_option("display.max_columns", None)  # to display all columns, without any limit
pd.set_option("display.max_colwidth", 100)  # to ensure that long text inside the cells won't get cut off



# 1. Load the cleaned dataset
The cleaned dataset produced during the data-cleaning stage is used as the input for this notebook. This avoids repeating the cleaning operations and ensures that preprocessing is performed on the same data used during EDA.

In [53]:
df = pd.read_csv("cleaned.csv")

print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (235370, 26)


,URL,Domain,URLLength,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,is_phishing
0,https://www.southbankmosaics.com,www.southbankmosaics.com,31,24,0,com,100.0,1.000000,0.522907,0.061933,3,1,0,0,0.0,18,0.581,0,0.0,0,0,0,1,0.032,1,0
1,https://www.uni-mainz.de,www.uni-mainz.de,23,16,0,de,100.0,0.666667,0.032650,0.050207,2,1,0,0,0.0,9,0.391,0,0.0,0,0,0,2,0.087,1,0
2,https://www.voicefmradio.co.uk,www.voicefmradio.co.uk,29,22,0,uk,100.0,0.866667,0.028555,0.064129,2,2,0,0,0.0,15,0.517,0,0.0,0,0,0,2,0.069,1,0
3,https://www.sfnmjournal.com,www.sfnmjournal.com,26,19,0,com,100.0,1.000000,0.522907,0.057606,3,1,0,0,0.0,13,0.500,0,0.0,0,0,0,1,0.038,1,0
4,https://www.rewildingargentina.org,www.rewildingargentina.org,33,26,0,org,100.0,1.000000,0.079963,0.059441,3,1,0,0,0.0,20,0.606,0,0.0,0,0,0,1,0.030,1,0


# 2. Dataset validation
Before extracting registered domains, the dataset is checked for missing values, duplicate URLs, inconsistent target values, and unexpected data types.

These checks help ensure that subsequent preprocessing is performed on valid and consistent data.

In [54]:
# Check required columns
required_columns = ["URL", "TLD", "is_phishing"]

for col in required_columns:
    assert col in df.columns, f"Missing required column: {col}"

# Check missing values
print("Missing values:", df[required_columns].isnull().sum())

# Check duplicate URLs
print("\nDuplicate URLs:", df["URL"].duplicated().sum())

# Check target values
print("\nTarget distribution:", df["is_phishing"].value_counts(dropna=False))

# Check invalid target values
print("\nUnexpected target values:", set(df["is_phishing"].dropna().unique()) - {0, 1})

Missing values: URL            0
TLD            0
is_phishing    0
dtype: int64

Duplicate URLs: 0

Target distribution: is_phishing
0    134850
1    100520
Name: count, dtype: int64

Unexpected target values: set()


# 3. Standardize categorical values
The TLD column is standardized by removing surrounding whitespace and converting values to lowercase.

This ensures that equivalent categorical values are represented consistently. The original URL is preserved because URL paths can be case-sensitive.

In [55]:
df["TLD"] = (
    df["TLD"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print("TLD standardization completed.")

print(df["TLD"].value_counts().head(10))

TLD standardization completed.
TLD
com    112382
org     18792
net      7076
app      6467
uk       6395
co       5408
io       4174
de       3994
ru       3870
au       2979
Name: count, dtype: Int64


# 4. Registered-domain extraction
A registered domain is extracted from each URL using the Public Suffix List.

Subdomains are removed so that URLs belonging to the same registered domain can be grouped together.

Private suffixes are included to distinguish independently hosted websites on shared hosting platforms where the relevant entries are available in the Public Suffix List.

The extracted domain will later be used as a grouping variable when creating the domain-disjoint train/test split.

In [56]:
# Include private suffixes in domain extraction
extractor = tldextract.TLDExtract(
    include_psl_private_domains=True
    # tells tldextract to consider private public-suffix entries, such as certain hosting platforms, when identifying registered domains.
)


def extract_registered_domain(url):

    if pd.isna(url) or not str(url).strip():
        return pd.NA

    url = str(url).strip()

    # Extract hostname
    parsed = urlsplit(
        url if "://" in url else "//" + url
    )

    hostname = parsed.hostname

    if not hostname:
        return pd.NA

    hostname = hostname.lower().rstrip(".")

    # Handle IP-address URLs
    try:
        return ipaddress.ip_address(hostname).compressed
    except ValueError:
        pass

    # Extract registered domain
    extracted = extractor(hostname)

    if extracted.domain and extracted.suffix:
        return f"{extracted.domain}.{extracted.suffix}"

    # Preserve the full hostname if no recognized suffix exists
    return hostname


# Apply extraction
df["registered_domain"] = df["URL"].apply(
    extract_registered_domain
)

print("Registered-domain extraction completed.")

display(
    df[["URL", "registered_domain"]].head(10)
)



def is_ip_address(value):
    if pd.isna(value):
        return False

    try:
        ipaddress.ip_address(value)
        return True
    except ValueError:
        return False


ip_domains = df[
    df["registered_domain"].apply(is_ip_address)
]

print("URLs using IP addresses:", len(ip_domains))

display(ip_domains[["URL", "registered_domain"]].head(10))

Registered-domain extraction completed.


,URL,registered_domain
0,https://www.southbankmosaics.com,southbankmosaics.com
1,https://www.uni-mainz.de,uni-mainz.de
2,https://www.voicefmradio.co.uk,voicefmradio.co.uk
3,https://www.sfnmjournal.com,sfnmjournal.com
4,https://www.rewildingargentina.org,rewildingargentina.org
5,https://www.globalreporting.org,globalreporting.org
6,https://www.saffronart.com,saffronart.com
7,https://www.nerdscandy.com,nerdscandy.com
8,https://www.hyderabadonline.in,hyderabadonline.in
9,https://www.aap.org,aap.org


URLs using IP addresses: 606


,URL,registered_domain
125,http://198.98.58.123/,198.98.58.123
401,http://43.134.167.94/servicelogin?passive=1209600&amp;continue=https://accounts.google.com/?&amp...,43.134.167.94
1205,http://43.128.92.128/servicelogin?passive=1209600&amp;continue=https://accounts.google.com/?&amp...,43.128.92.128
1362,http://43.134.167.94/v3/signin/identifier?dsh=s151927330%3a1679418096294349&amp%3bfollowup=https...,43.134.167.94
2481,http://43.153.207.103/v3/signin/identifier?dsh=s-16339560%3a1679418180081355&amp%3bfollowup=http...,43.153.207.103
2530,http://43.156.0.130/servicelogin?passive=1209600&amp;continue=https://accounts.google.com/&amp;f...,43.156.0.130
5658,http://134.122.69.160/kirs/yes.php?logx2&amp;open=757662,134.122.69.160
5705,http://43.134.191.197/v3/signin/identifier?dsh=s-1106407172:1663063518048988&amp;continue=https:...,43.134.191.197
6338,http://81.19.136.232/,81.19.136.232
8788,http://43.128.92.128/interactivelogin?continue=https://accounts.google.com/?&amp;xrealip=35.203....,43.128.92.128


# 5. Validate registered domains
After extraction, the resulting registered-domain column is checked for missing or empty values and the total number of unique domains is calculated.

IP-address URLs are also examined separately to verify that they have been handled correctly.

In [57]:
print(
    "Missing registered domains:",
    df["registered_domain"].isna().sum()
)

print(
    "Empty registered domains:",
    (df["registered_domain"] == "").sum()
)

print(
    "Unique registered domains:",
    df["registered_domain"].nunique()
)

Missing registered domains: 0
Empty registered domains: 0
Unique registered domains: 197301


# 6. Domain repetition and class overlap analysis
This analysis examines how frequently registered domains occur in the dataset and whether the same domain appears with both legitimate and phishing labels.

Repeated domains can create information leakage in conventional random train/test splits, as URLs from the same domain may appear in both sets.

The analysis provides evidence for the need to evaluate models using a domain-disjoint split.

In [58]:
# Group records by registered domain
domain_stats = (
    df.groupby("registered_domain")["is_phishing"]
    .agg(
        n_urls="count",
        n_classes="nunique"
    )
)

# Total unique domains
print(
    "Unique domains:",
    len(domain_stats)
)

# Domains appearing in both classes
both_classes = domain_stats[
    domain_stats["n_classes"] == 2
]

print(
    "Domains appearing in BOTH classes:",
    len(both_classes)
)

# Domains with repeated URLs
repeated_domains = domain_stats[
    domain_stats["n_urls"] > 1
]

print(
    "Domains with more than one URL:",
    len(repeated_domains)
)

# Percentage of dataset rows belonging to repeated domains
repeated_rows = repeated_domains["n_urls"].sum()

percentage = (
    repeated_rows / len(df)
) * 100

print(
    "Percentage of rows from repeated domains:",
    round(percentage, 2),
    "%"
)

Unique domains: 197301
Domains appearing in BOTH classes: 106
Domains with more than one URL: 7463
Percentage of rows from repeated domains: 19.34 %


# 7. Most frequently occurring registered domains
The most frequently occurring registered domains are identified to examine whether a small number of domains contribute disproportionately to the dataset.

Their class distributions are also examined to identify domains associated with only one class or appearing in both classes.

In [59]:
# Top 20 domains by URL count
top_domains = (
    domain_stats["n_urls"]
    .sort_values(ascending=False)
    .head(20)
)

print("Top 20 registered domains:")
display(top_domains)

# Class distribution for these domains
top_domain_rows = df[
    df["registered_domain"].isin(top_domains.index)
]

domain_label_distribution = pd.crosstab(
    top_domain_rows["registered_domain"],
    top_domain_rows["is_phishing"]
)

domain_label_distribution = (
    domain_label_distribution
    .reindex(top_domains.index)
    .fillna(0)
)

domain_label_distribution.columns = [
    "Legitimate (0)",
    "Phishing (1)"
]

display(domain_label_distribution)

Top 20 registered domains:


,n_urls
registered_domain,
weeblysite.com,3079
ipfs.io,1552
xsph.ru,928
google.com,782
godaddysites.com,770
webwave.dev,559
weebly.com,552
glitch.me,509
fleek.co,468


,Legitimate (0),Phishing (1)
registered_domain,,
weeblysite.com,0,3079
ipfs.io,0,1552
xsph.ru,0,928
google.com,0,782
godaddysites.com,0,770
webwave.dev,0,559
weebly.com,0,552
glitch.me,0,509
fleek.co,0,468


# 8. Save the processed dataset
The registered-domain column is added to the cleaned dataset, and the resulting dataset is saved for use in subsequent experiments.

The original URL and feature columns are preserved. The registered-domain column will serve as a grouping variable during domain-disjoint evaluation.

In [60]:
# Create output directory
os.makedirs("data/processed", exist_ok=True)

# Save the complete processed dataset
output_path = (
    "data/processed/cleaned_with_domains.csv.gz"
)

df.to_csv(
    output_path,
    index=False,
    compression="gzip"
)

print("Processed dataset saved successfully.")

print("Final shape:", df.shape)

print(
    "File size:",
    round(os.path.getsize(output_path) / (1024 ** 2), 2),
    "MB"
)

Processed dataset saved successfully.
Final shape: (235370, 27)
File size: 9.83 MB


# 9. Scaling and categorical encoding decisions
The numerical features have different ranges and scales. Standardization will therefore be considered for Logistic Regression.

Tree-based models such as Random Forest and XGBoost do not generally require numerical feature scaling.

The TLD column is retained as a categorical feature. It will be encoded using an appropriate categorical encoding technique during model development.

Scaling and encoding will be fitted using training data only after the train/test split to prevent data leakage. Unseen TLD categories will be handled during model evaluation.

# 10. Key findings from domain analysis
1. **Domain diversity:** The dataset contains 197,301 unique registered domains across 235,370 URLs. This means the dataset contains a large number of distinct domains.

2. **Repeated domains:** 7,463 registered domains occur more than once, accounting for 19.34% of all dataset records. This matters because a conventional random split could put URLs from the same domain in both training and testing sets. A model might perform well partly because it has already encountered URLs from those domains during training.

3. **Cross-class domain overlap:** 106 domains appear with both legitimate and phishing labels. This is an important finding: a domain alone does not always determine whether a URL is legitimate or phishing. It also supports evaluating the model on unseen domains rather than relying exclusively on random splitting.

4. **Frequently occurring domains:** Several of the most frequently occurring domains appear exclusively with phishing labels in the dataset. For example, weeblysite.com accounts for 3,079 phishing URLs, while ipfs.io accounts for 1,552. This suggests potential domain-related patterns in the dataset, which should be examined further during model evaluation.

5. **Evaluation implication:** Repeated domains can introduce domain overlap between training and testing sets under conventional random splitting. Domain-disjoint evaluation is therefore important for measuring generalization to previously unseen domains.